## KGAT & KGATSAGE

In [ ]:
import pandas as pd
import numpy as np
import argparse
import random
from KGATModel import KGAT
from improved_data_loader import DataLoader
import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## Hyperparameters Settings

In [2]:
# prepare arguments (hyperparameters)
parser = argparse.ArgumentParser()

parser.add_argument('--dataset', type=str, default='movie', help='which dataset to use')
parser.add_argument('--aggregator', type=str, default='sum', help='which aggregator to use')
parser.add_argument('--n_epochs', type=int, default=20, help='the number of epochs')
parser.add_argument('--neighbor_sample_size', type=int, default=8, help='the number of neighbors to be sampled')
parser.add_argument('--dim', type=int, default=16, help='dimension of user and entity embeddings')
parser.add_argument('--n_iter', type=int, default=1, help='number of iterations when computing entity representation')
parser.add_argument('--batch_size', type=int, default=32, help='batch size')
parser.add_argument('--l2_weight', type=float, default=1e-4, help='weight of l2 regularization')
parser.add_argument('--lr', type=float, default=5e-4, help='learning rate')
parser.add_argument('--ratio', type=float, default=0.8, help='size of training dataset')

args = parser.parse_args(['--l2_weight', '1e-4', '--lr', '1e-3'])

## Loading the knowledge graph and the ratings dataset

In [3]:
# build dataset and knowledge graph
data_loader = DataLoader(args.dataset)
kg = data_loader.load_kg()
df_dataset, users_dataframe = data_loader.load_dataset()
df_dataset

Construct knowledge graph ... Done
Build dataset dataframe ... Done


,userID,itemID,rating
0,0,742,5
1,0,433,3
2,0,580,3
3,0,2107,4
4,0,1432,5
...,...,...,...
664220,6039,671,1
664221,6039,674,5
664222,6039,370,5
664223,6039,676,4


In [4]:
users_dataframe

,userID,Gender,Age,Occupation_0,Occupation_1,Occupation_2,Occupation_3,Occupation_4,Occupation_5,Occupation_6,...,Occupation_11,Occupation_12,Occupation_13,Occupation_14,Occupation_15,Occupation_16,Occupation_17,Occupation_18,Occupation_19,Occupation_20
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,56,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,2,1,25,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,3,1,45,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,1,25,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6035,6035,0,25,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
6036,6036,0,45,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6037,6037,0,56,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6038,6038,0,45,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Data Loader

In [ ]:
# Dataset class
class KGATDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        user_id = torch.from_numpy(np.array(users_dataframe[users_dataframe['userID'] == self.df.iloc[idx]['userID']].values[0], dtype=np.float32))
        item_id = torch.from_numpy(np.array(self.df.iloc[idx]['itemID']))
        rating = torch.from_numpy(np.array(self.df.iloc[idx]['rating'], dtype=np.float32))
        return user_id, item_id, rating

## Training Setup 

In [ ]:
# train test split
x_train, x_test = train_test_split(df_dataset, test_size=1 - args.ratio, shuffle=False, random_state=999)
train_dataset = KGATDataset(x_train)
test_dataset = KGATDataset(x_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=args.batch_size)

def RMSELoss(outouts, ratings):
    mse_loss = torch.nn.MSELoss()
    return torch.sqrt(mse_loss(outouts, ratings))

def train(net, optimizer, criterion, device, train_loader, test_loader):
    loss_list = []
    test_loss_list = []
    for epoch in range(args.n_epochs):
        running_loss = 0.0
        net.train()
        for i, (user_ids, item_ids, ratings) in enumerate(train_loader):
            user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
            optimizer.zero_grad()
            outputs = net(user_ids, item_ids)
            outputs = outputs.flatten()
            loss = criterion(outputs, ratings)
            loss.backward()
            
            optimizer.step()

            running_loss += loss.item()
    
        # print train loss per every epoch
        print('[Epoch {}]train_loss: '.format(epoch+1), running_loss / len(train_loader))
        loss_list.append(running_loss / len(train_loader))
        
        # evaluate per every epoch
        net.eval()
        with torch.no_grad():
            test_loss = 0
            #total_acc = 0
            for user_ids, item_ids, ratings in test_loader:
                user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
                outputs = net(user_ids, item_ids)
                outputs = torch.clip(outputs, 1, 5)
                outputs = outputs.flatten()
                test_loss += criterion(outputs, ratings).item()
            print('[Epoch {}]test_loss: '.format(epoch+1), test_loss / len(test_loader))
            test_loss_list.append(test_loss / len(test_loader))
    return loss_list, test_loss_list    

## Training KGAT Model

In [26]:
random_seed = 49
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
num_user, num_entity, num_relation = data_loader.get_num()
user_encoder, entity_encoder, relation_encoder = data_loader.get_encoders()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
kgat = KGAT(num_user, num_entity, num_relation, kg, args, device).to(device)
criterion = RMSELoss
optimizer = optim.Adam(kgat.parameters(), lr=0.001, weight_decay=args.l2_weight)

kgat_loss_list, kgat_test_loss_list = train(kgat, optimizer, RMSELoss, device, train_loader, test_loader)


[Epoch 1]train_loss:  1.1114830260361215
[Epoch 1]test_loss:  1.0136469723315826
[Epoch 2]train_loss:  1.0067965567352455
[Epoch 2]test_loss:  1.009964722702154
[Epoch 3]train_loss:  1.0019900805527626
[Epoch 3]test_loss:  1.0027716519788397
[Epoch 4]train_loss:  0.9889738752201467
[Epoch 4]test_loss:  0.9936784051775014
[Epoch 5]train_loss:  0.9868311388376577
[Epoch 5]test_loss:  0.9897249791275904
[Epoch 6]train_loss:  0.9816845946608861
[Epoch 6]test_loss:  0.9888506255799865
[Epoch 7]train_loss:  0.9806601025391153
[Epoch 7]test_loss:  0.9842649922061977
[Epoch 8]train_loss:  0.9778961691564355
[Epoch 8]test_loss:  0.980220248057778
[Epoch 9]train_loss:  0.9773942594494172
[Epoch 9]test_loss:  0.9794854107734555
[Epoch 10]train_loss:  0.9771041320175862
[Epoch 10]test_loss:  0.9765071666338311
[Epoch 11]train_loss:  0.976944870782892
[Epoch 11]test_loss:  0.9762719524001455
[Epoch 12]train_loss:  0.9763573308002083
[Epoch 12]test_loss:  0.9783239321915975
[Epoch 13]train_loss:  0.

In [27]:
torch.save(kgat.state_dict(), '../checkpoints/KGAT.pth')

## Training KGATSAGE Model

In [ ]:
args.aggregator = 'concat'
random_seed = 49
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
num_user, num_entity, num_relation = data_loader.get_num()
user_encoder, entity_encoder, relation_encoder = data_loader.get_encoders()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
kgatsage = KGAT(num_user, num_entity, num_relation, kg, args, device).to(device)
criterion = RMSELoss
optimizer = optim.Adam(kgatsage.parameters(), lr=0.001, weight_decay=args.l2_weight)

kgat_loss_list, kgat_test_loss_list = train(kgatsage, optimizer, RMSELoss, device, train_loader, test_loader)


[Epoch 1]train_loss:  1.0967712601767696
[Epoch 1]test_loss:  1.00541869825854
[Epoch 2]train_loss:  1.0057314190500748
[Epoch 2]test_loss:  1.0077630468645997
[Epoch 3]train_loss:  1.0023821491990823
[Epoch 3]test_loss:  1.0052470473827424
[Epoch 4]train_loss:  1.0009912602602538
[Epoch 4]test_loss:  1.005589030905553
[Epoch 5]train_loss:  0.996473896500577
[Epoch 5]test_loss:  0.9938477631981317
[Epoch 6]train_loss:  0.9859245909088766
[Epoch 6]test_loss:  0.9904962732009805
[Epoch 7]train_loss:  0.9855054799518025
[Epoch 7]test_loss:  0.992003315108647
[Epoch 8]train_loss:  0.9836804222162554
[Epoch 8]test_loss:  0.9857498546182994
[Epoch 9]train_loss:  0.9808063535543575
[Epoch 9]test_loss:  0.9870120365825806
[Epoch 10]train_loss:  0.980733167871386
[Epoch 10]test_loss:  0.9839185059558679
[Epoch 11]train_loss:  0.9802050109973776
[Epoch 11]test_loss:  0.9842702554743414
[Epoch 12]train_loss:  0.9807044381346887
[Epoch 12]test_loss:  0.9851774686951054
[Epoch 13]train_loss:  0.978

In [ ]:
torch.save(kgat.state_dict(), '../checkpoints/KGATSAGE.pth')